In [1]:
!pip install pypdf

# dotenv load_dotenv

In [2]:
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from pinecone import Pinecone, ServerlessSpec

load_dotenv(override=True)


c:\Users\Admin\miniconda3\envs\langchain_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
# pypdf 설치 확인
try:
    import pypdf
    print(f"pypdf: {pypdf.__version__}")
except ImportError:
    print("pypdf 미설치 → pip install pypdf")

pypdf: 6.9.2


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# PDF 로드

In [5]:
PDF_PATH = "data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f"총 페이지 수: {len(docs)}")


총 페이지 수: 93


# 인덱스 신규 생성

In [6]:
pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

INDEX_NAME = "finance-bok-test"

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"✅ {INDEX_NAME} 인덱스 생성 완료")
else:
    print(f"이미 존재함: {INDEX_NAME}")

test_index = pc.Index(INDEX_NAME)
print(test_index.describe_index_stats())


✅ finance-bok-test 인덱스 생성 완료
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


# 3가지 청크 전략 Splitter 정의

In [7]:
# 전략 A — 작은 청크 (정밀 검색)
splitter_a = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)

# 전략 B — 중간 청크 (권장)
splitter_b = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

# 전략 C — 큰 청크 (문맥 유지)
splitter_c = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

# 청크 수 비교 출력
for name, splitter in [("A(200)", splitter_a), ("B(500)", splitter_b), ("C(1000)", splitter_c)]:
    splits = splitter.split_documents(docs)
    avg_len = sum(len(s.page_content) for s in splits) // len(splits)
    print(f"전략 {name}: 청크 수 {len(splits)}개 | 평균 길이 {avg_len}자")


전략 A(200): 청크 수 581개 | 평균 길이 163자
전략 B(500): 청크 수 249개 | 평균 길이 400자
전략 C(1000): 청크 수 145개 | 평균 길이 688자


# 임베딩 모델 생성

In [8]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
print("임베딩 모델 준비 완료")

임베딩 모델 준비 완료


# 3가지 Namespace 각각 업서트

In [9]:
# Namespace 이름 정의
NS_A = "chunk-200"
NS_B = "chunk-500"
NS_C = "chunk-1000"

strategies = [
    ("A(200)",  splitter_a, NS_A),
    ("B(500)",  splitter_b, NS_B),
    ("C(1000)", splitter_c, NS_C),
]

vectorstores = {}

for name, splitter, namespace in strategies:
    splits = splitter.split_documents(docs)
    print(f"[{name}] 업서트 시작 — {len(splits)}개 청크 → namespace: {namespace}")
    vs = PineconeVectorStore.from_documents(
        documents=splits,
        embedding=embedding_model,
        index_name=INDEX_NAME,
        namespace=namespace
    )
    vectorstores[name] = vs
    print(f"[{name}] ✅ 업서트 완료\n")

print("전체 업서트 완료")


[A(200)] 업서트 시작 — 581개 청크 → namespace: chunk-200
[A(200)] ✅ 업서트 완료

[B(500)] 업서트 시작 — 249개 청크 → namespace: chunk-500
[B(500)] ✅ 업서트 완료

[C(1000)] 업서트 시작 — 145개 청크 → namespace: chunk-1000
[C(1000)] ✅ 업서트 완료

전체 업서트 완료


# 업서트 결과 확인

In [10]:
stats = test_index.describe_index_stats()
print(stats)

# namespace별 벡터 수 확인
print("\n[ Namespace별 벡터 수 ]")
for ns, info in stats['namespaces'].items():
    print(f"  {ns}: {info['vector_count']}개")


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'chunk-1000': {'vector_count': 145},
                'chunk-200': {'vector_count': 581},
                'chunk-500': {'vector_count': 249}},
 'total_vector_count': 975,
 'vector_type': 'dense'}

[ Namespace별 벡터 수 ]
  chunk-1000: 145개
  chunk-200: 581개
  chunk-500: 249개


# 3가지 Retriever 생성

In [11]:
retriever_a = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    namespace=NS_A
).as_retriever(search_kwargs={"k": 3})

retriever_b = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    namespace=NS_B
).as_retriever(search_kwargs={"k": 3})

retriever_c = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    namespace=NS_C
).as_retriever(search_kwargs={"k": 3})

print("retriever A / B / C 준비 완료")


retriever A / B / C 준비 완료


# 검색 품질 비교 (retriever 평가)

In [12]:
test_query = "2026년 GDP 성장률 전망치는?"

print(f"[ 질문 ] {test_query}\n")

for name, retriever in [("A 작은청크(200)", retriever_a),
                         ("B 중간청크(500)", retriever_b),
                         ("C 큰청크(1000)", retriever_c)]:
    print(f"{'='*60}")
    print(f"전략 {name}")
    results = retriever.invoke(test_query)
    for i, r in enumerate(results):
        print(f"  {i+1}. (p.{r.metadata.get('page','')}) {r.page_content[:150]}")
    print()


[ 질문 ] 2026년 GDP 성장률 전망치는?

전략 A 작은청크(200)
  1. (p.43.0) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 1.6 1.0 2.4 1.6 2.0 1.8 
 <0.3> <1.8> <1.0> <2.2> 
  2. (p.79.0) 66 
 
한국은행 전망에서는 점진적∙보수적인 전망 경향이 나타남  
[그림3] 분기별 GDP 성장률1) 및 전망 [그림4] 분기성장률(전년동기대비) 증감과 전망오차2) 
  
   주: 1) 속보치 기준         2) 성장률이 전기보다 확대되는 경우 과소추정, 
  3. (p.13.0) <국내경제 전망>           
GDP 성장률(%)3) 2.0  0.3 1.6 1.0[ - ] ..  2.4 1.6 2.0[+0.2]  1.8[-0.1] 
• 민간소비 1.1  0.7 1.9 1.3[ - ] ..  2.3 1.3 1.8[+0.1]  1.8[+0.1

전략 B 중간청크(500)
  1. (p.43.0) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 1.6 1.0 2.4 1.6 2.0 1.8 
 <0.3> <1.8> <1.0> <2.2> 
  2. (p.79.0) 66 
 
한국은행 전망에서는 점진적∙보수적인 전망 경향이 나타남  
[그림3] 분기별 GDP 성장률1) 및 전망 [그림4] 분기성장률(전년동기대비) 증감과 전망오차2) 
  
   주: 1) 속보치 기준         2) 성장률이 전기보다 확대되는 경우 과소추정, 
  3. (p.9.0) ▪ 27년에는 내수 회복세가 지속되는 가운데 수출도 세계경제 성장세 지속, 반도체 
공급능력 확충 등으로 증가하며 1.8%의 견조한 성장세를 나타낼 전망이다. 
 
 
 
<국내 성장률

# RAG Chain 3개 만들어서 답변 품질 비교 (Generation 평가)

In [13]:
prompt = ChatPromptTemplate.from_template("""
당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하세요.
""")

llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

def make_chain(retriever):
    return (
        RunnableParallel(context=retriever, question=RunnablePassthrough())
        | prompt | llm | parser
    )

chain_a = make_chain(retriever_a)
chain_b = make_chain(retriever_b)
chain_c = make_chain(retriever_c)

# 비교 질문 3개
questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"[ Q ] {q}")
    print(f"{'-'*60}")
    print(f"[A 작은청크 200] {chain_a.invoke(q)}")
    print(f"[B 중간청크 500] {chain_b.invoke(q)}")
    print(f"[C 큰청크 1000] {chain_c.invoke(q)}")



[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
------------------------------------------------------------
[A 작은청크 200] 2026년 GDP 성장률 전망치는 2.0%입니다.
[B 중간청크 500] 2026년 GDP 성장률 전망치는 2.0%입니다.
[C 큰청크 1000] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
------------------------------------------------------------
[A 작은청크 200] 소비자물가 상승률은 전자기기 및 일부 서비스 가격 인상압력 등으로 당초 전망경로를 소폭 웃돌 것으로 전망됩니다. 2024년부터 2027년까지의 소비자물가 상승률은 각각 2.3%, 2.1%, 2.2%, 2.1%로 예상됩니다.
[B 중간청크 500] 소비자물가 상승률은 2026년 2.0%로 전망되고 있습니다. 이는 11월 전망인 1.9% 대비 0.1%p 높아진 수치입니다.
[C 큰청크 1000] 소비자물가 상승률은 2026년 2.2%로 전망되며, 2027년에는 2.0%로 예상됩니다.

[ Q ] 수출 전망은 어떻게 되나요?
------------------------------------------------------------
[A 작은청크 200] 보고서에서 확인되지 않습니다.
[B 중간청크 500] 수출은 견조한 흐름을 이어갈 것으로 전망되며, 2024년에는 6,836억 달러, 2025년에는 7,093억 달러, 2026년에는 7,952억 달러로 증가할 것으로 예상됩니다.
[C 큰청크 1000] 2024년 수출 전망은 6,836억 달러로, 전년 동기 대비 8.1% 증가할 것으로 예상됩니다. 2025년에는 7,093억 달러로 3.8% 증가할 것으로 보이며, 2026년에는 7,952억 달러로 12.1% 증가할 것으로 전망됩니다. 2027년에는 7,804억 달러로 -1.9% 감소할 것으로 예상됩니다.
